# Lab 1.1 — Anatomy of a Prompt

We take one extraction task and build the prompt up **one component at a time**:
instruction only → + role → + context → + output-format spec → + one example.

**Watch for:** each new component fixes a specific failure in the previous
output. By the end, you should be able to point at the exact prompt line
responsible for a given improvement.

## Cache redirect — point Hugging Face downloads at your NFS volume

The pod's local disk is small and can fill up fast once you're downloading
multi-GB model weights (this is what causes `disk is full` errors when
saving). Redirect the Hugging Face cache to your larger, persistent NFS
volume **before** any model is loaded.

**Edit the volume name below to match yours** (check with `ls ~` in a
terminal if you're not sure).

In [1]:
import os

# Redirect HF caches to the NFS volume instead of the pod's local disk —
# must run before transformers/torch download anything.
NFS_CACHE_ROOT = os.path.expanduser("~/llmall/.cache/huggingface")

os.environ["HF_HOME"] = os.path.join(NFS_CACHE_ROOT, "huggingface")
os.environ["HF_DATASETS_CACHE"] = os.path.join(NFS_CACHE_ROOT, "datasets")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.makedirs(os.environ["HF_DATASETS_CACHE"], exist_ok=True)

print("HF cache redirected to:", os.environ["HF_HOME"])

HF cache redirected to: /home/jovyan/llmall/.cache/huggingface/huggingface


## Setup — Local model via `transformers` (CPU-only, no API credits needed)

We're running a small model **locally** instead of calling a hosted API —
no per-request cost, and it works even if the Inference API is rate-limited
or out of credits.

**Kubeflow users:** do not request a GPU — these models are small enough
(0.5B–1.5B parameters) to run at acceptable speed on CPU for a live demo.

**Token:** not required for the models used here — they're public,
ungated repos. If you ever swap in a gated model, reuse the `HF_TOKEN`
setup from earlier and pass `token=HF_TOKEN` to `from_pretrained(...)`.

**If you see a `torch`/`torchvision`/`torchaudio` version-conflict warning**
during install, or a `torchvision::nms does not exist` error when loading
the model: these notebooks don't need `torchvision` or `torchaudio` at all.
Run this once and restart the kernel:
```bash
pip uninstall -y torchvision torchaudio
```

In [2]:
import sys
print(sys.executable)

/home/jovyan/llm2/venv/bin/python


In [3]:
!pip install --quiet -U transformers accelerate
!pip install --quiet torch  # only installs if missing -- avoids upgrading an existing torch and breaking any torchvision/torchaudio pinned to it
print("Packages installed/upgraded. If this is the first install this session, restart the kernel now, then run from the top.")

Packages installed/upgraded. If this is the first install this session, restart the kernel now, then run from the top.


In [4]:
import os
os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
torch.set_num_threads(6)
torch.set_num_interop_threads(1)

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading {MODEL} ... (first run downloads the weights)")
tokenizer = AutoTokenizer.from_pretrained(MODEL)

#MODEL LOADING (WEIGHTS + ARCHITECTURE)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32, device_map="cpu")

#SET INFERENCE MODE
model.eval()
print("Model ready.")

Loading Qwen/Qwen2.5-0.5B-Instruct ... (first run downloads the weights)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model ready.


In [6]:
def ask(messages, max_tokens=250, temperature=0.0):
    """Generate a reply from the locally loaded `model`/`tokenizer`."""

   #   DECIDE SAMPLING STRATEGY
    do_sample = temperature > 0

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt")

    #BUILD GENERATION CONFIG : Collects the arguments that control how generate() behaves.
    gen_kwargs = dict(
        max_new_tokens=max_tokens,
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
    )

    if do_sample:
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)

  #STRIP OFF THE INPUT PROMPT
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]

 #DECODE TOKENS BACK TO TEXT
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

## Task

A messy customer support message. We want to reliably pull out: product,
order number, the issue, and what resolution the customer wants — something
a downstream ticketing system could consume.

In [7]:
messy_message = """
Hi, so I ordered the SoundWave X2 earbuds last week, order #48213, and one of
them just stopped charging after 3 days!! I've tried different cables.
I'd like a replacement, not a refund. Please help fast, I need them for a
trip on Friday.
"""
print(messy_message)


Hi, so I ordered the SoundWave X2 earbuds last week, order #48213, and one of
them just stopped charging after 3 days!! I've tried different cables.
I'd like a replacement, not a refund. Please help fast, I need them for a
trip on Friday.



### Stage A — Instruction only

In [8]:
prompt_a = f"Extract the key details from this message:\n\n{messy_message}"
out_a = ask([{"role": "user", "content": prompt_a}], max_tokens=60) #prompt
print(out_a)

/home/jovyan/llm2/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/home/jovyan/llm2/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/home/jovyan/llm2/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


The key details extracted from the message include:

- The customer's name: Hi
- The product they purchased: SoundWave X2 earbuds
- The order number: Order #48213
- The issue encountered: One earbud stopped charging after 3 days
-


**Expect:** a rambling free-text summary — plausible-sounding, but not
something you could parse programmatically, and it may miss or bury a field.

### Stage B — + Role

In [9]:
prompt_b = (
    "You are a customer support ticket triager who extracts precise, "
    "structured details from complaints.\n\n"
    f"Extract the key details from this message:\n\n{messy_message}"
)
out_b = ask([{"role": "user", "content": prompt_b}])
print(out_b)

Certainly! Here is the extracted key information from your complaint:

- **Order Number:** 48213
- **Issue:** The earbuds stopped charging after 3 days
- **Solution Needed:** Replacement earbuds
- **Time Frame:** For a trip on Friday
- **Customer Needs:** Fast response

This summary captures all the essential details you provided in a concise manner.


**Expect:** tone/framing shifts toward "ticket-like" language, but the
*shape* of the output usually still isn't fixed — role alone rarely fixes
structure.

### Stage C — + Context (why these fields matter)

In [10]:
prompt_c = (
    "You are a customer support ticket triager who extracts precise, "
    "structured details from complaints.\n\n"
    "Context: these extracted details feed an automated routing system. "
    "We specifically need: the product name, the order number, a one-line "
    "issue description, and the customer's requested resolution "
    "(replacement / refund / other).\n\n"
    f"Extract the key details from this message:\n\n{messy_message}"
)
out_c = ask([{"role": "user", "content": prompt_c}])
print(out_c)

Sure, here is the extracted key information:

- **Product Name:** SoundWave X2 earbuds
- **Order Number:** 48213
- **Issue Description:** The earbuds stopped charging after 3 days
- **Customer Requested Resolution:** Replacement
- **Reason for Issue:** Not a refund

This information should be sufficient to guide the customer service representative in providing the appropriate assistance.


**Expect:** the four fields are now reliably present and named
correctly — but the format is still prose or an ad-hoc list, not something
you can `json.loads()`.

### Stage D — + Output-format spec

In [11]:
prompt_d = (
    "You are a customer support ticket triager who extracts precise, "
    "structured details from complaints.\n\n"
    "Context: these extracted details feed an automated routing system.\n\n"
    "Return ONLY a JSON object with exactly these keys:\n"
    "{\"product\": string, \"order_number\": string, \"issue\": string, "
    "\"requested_resolution\": string}\n\n"
    f"Message:\n{messy_message}"
)
out_d = ask([{"role": "user", "content": prompt_d}])
print(out_d)

{
  "product": "SoundWave X2",
  "order_number": "48213",
  "issue": "Earbuds stopped charging",
  "requested_resolution": "Replacement"
}


**Expect:** valid-looking JSON with the right keys. This is usually
"good enough" — but watch the exact wording of `requested_resolution`: does
it output `"replacement"` cleanly, or does it echo the customer's sentence?
This is the gap Stage E targets.

### Stage E — + One example (single-shot)

In [12]:
example_message = "The blender arrived cracked, order #77120. I just want my money back."
example_output = (
    '{"product": "blender", "order_number": "77120", '
    '"issue": "arrived cracked", "requested_resolution": "refund"}'
)

prompt_e_system = (
    "You are a customer support ticket triager who extracts precise, "
    "structured details from complaints.\n\n"
    "Context: these extracted details feed an automated routing system.\n\n"
    "Return ONLY a JSON object with exactly these keys:\n"
    "{\"product\": string, \"order_number\": string, \"issue\": string, "
    "\"requested_resolution\": string}\n\n"
    "requested_resolution must be one of: replacement, refund, other."
)

messages_e = [
    {"role": "system", "content": prompt_e_system},
    {"role": "user", "content": example_message},
    {"role": "assistant", "content": example_output},
    {"role": "user", "content": messy_message},
]
out_e = ask(messages_e)
print(out_e)

{"product": "SoundWave X2 earbuds", "order_number": "48213", "issue": "charging issue", "requested_resolution": "replacement"}


**Expect:** `requested_resolution` now snaps to a clean label
(`"replacement"`) matching the example's style, rather than restating the
customer's phrasing.

## Side-by-side comparison

In [13]:
import json

results = {"A (instruction only)": out_a, "B (+role)": out_b,
           "C (+context)": out_c, "D (+format)": out_d, "E (+example)": out_e}

for stage, output in results.items():
    print(f"--- {stage} ---")
    print(output)
    print()

--- A (instruction only) ---
The key details extracted from the message include:

- The customer's name: Hi
- The product they purchased: SoundWave X2 earbuds
- The order number: Order #48213
- The issue encountered: One earbud stopped charging after 3 days
-

--- B (+role) ---
Certainly! Here is the extracted key information from your complaint:

- **Order Number:** 48213
- **Issue:** The earbuds stopped charging after 3 days
- **Solution Needed:** Replacement earbuds
- **Time Frame:** For a trip on Friday
- **Customer Needs:** Fast response

This summary captures all the essential details you provided in a concise manner.

--- C (+context) ---
Sure, here is the extracted key information:

- **Product Name:** SoundWave X2 earbuds
- **Order Number:** 48213
- **Issue Description:** The earbuds stopped charging after 3 days
- **Customer Requested Resolution:** Replacement
- **Reason for Issue:** Not a refund

This information should be sufficient to guide the customer service representat

## Discussion

identify, stage by stage, **which specific failure was
fixed** by the new component:

| Stage added | Typical failure it fixes |
|---|---|
| Role | Tone / framing, rarely structure |
| Context | Missing or wrong fields |
| Output format | Unparseable output → valid JSON shape |
| Example | Label consistency / exact wording (the JSON *mode* guarantees syntax, but only the example nails the exact vocabulary) |

**Takeaway:** each component does a distinct job. Adding all five doesn't
guarantee correctness but it collapses most of the *ambiguity* in what the model should even
attempt.

## Practice — try it yourself

Same five-stage idea, applied to new messages. Re-use the Stage E
system prompt (`prompt_e_system`) and one-shot example as your fixed
format/schema, and see how it holds up across messages that are messier
in different ways than the original one.

practice_messages = {
    "blender": (
        "Hey, the blender I got, order #91042, showed up with the lid "
        "cracked right down the middle. Water leaks everywhere when I "
        "blend anything. Can you just send a new one? Don't really want "
        "to deal with a refund and re-ordering."
    ),
    "headphones": (
        "order 55310 - the noise cancelling on these headphones doesn't "
        "work at all, might as well not have it. kind of a big reason I "
        "bought them tbh. at this point I'd rather just get my money back."
    ),
    "keyboard": (
        "Ordered the mechanical keyboard (order #10287) two weeks ago and "
        "three of the keys have already stopped registering keypresses. "
        "Not sure if I want a replacement or a refund honestly, whichever "
        "is faster for you."
    ),
    "no_order_number": (
        "My toaster is making a burning smell every time I use it and I'm "
        "honestly a little worried about it. I don't have my order number "
        "handy but it was bought sometime last month. Would like a "
        "replacement please."
    ),
}

def run_one_shot_extraction(message):
    """Runs the Stage E (system + example) prompt on a new message."""
    messages = [
        {"role": "system", "content": prompt_e_system},
        {"role": "user", "content": example_message},
        {"role": "assistant", "content": example_output},
        {"role": "user", "content": message},
    ]
    return ask(messages)

for name, msg in practice_messages.items():
    print(f"=== {name} ===")
    print(run_one_shot_extraction(msg))
    print()

### Things to look for while practicing

- **`headphones`** and **`keyboard`**: the customer doesn't clearly pick one
  resolution — does the model force a choice, ask for clarification, or
  guess?.
- **`no_order_number`**: no order number is given at all. Does the model
  invent one, leave it blank, or write `"unknown"`? Try tightening the
  schema instruction (e.g. `"order_number": "use null if not mentioned"`)
  and see if that stops it from fabricating a value.
- Try writing **one message of your own** that breaks a rule the others
  don't — two products in one message, or a customer who changes their
  mind mid-message — and see where the Stage E prompt holds up and where
  it doesn't.